In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import os
os.chdir("/content")
print(os.getcwd())


/content


In [3]:
!rm -rf /content/Task-driven-low-light-enhancement
!git clone https://github.com/SarthakBaghel/Task-driven-low-light-enhancement.git /content/Task-driven-low-light-enhancement
%cd /content/Task-driven-low-light-enhancement
!pip install -r requirements.txt


Cloning into '/content/Task-driven-low-light-enhancement'...
remote: Enumerating objects: 72, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 72 (delta 15), reused 67 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (72/72), 158.34 KiB | 1.21 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Task-driven-low-light-enhancement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.3 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Switch runtime to GPU for faster evaluation.")


CUDA available: True
GPU: Tesla T4


In [5]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
PIPELINE_ROOT = DRIVE_ROOT / "task_driven_video_pipeline"
KAGGLE_V1_ROOT = PIPELINE_ROOT / "kaggle_v1"
CHECKPOINTS_ROOT = DRIVE_ROOT / "task_driven_checkpoints" / "kaggle_v1"

RUN_TAG = "subject_class_balanced_20k"
LOWLIGHT_TAG = "eye_mid"

MAX_TOTAL_SAMPLES = 5000
NEW_SUBSET_SEED = 314
BATCH_SIZE = 64
NUM_WORKERS = 0

TEST_CLEAN_ROOT = KAGGLE_V1_ROOT / f"test_clean_{RUN_TAG}"
TEST_LOWLIGHT_ROOT = KAGGLE_V1_ROOT / f"test_lowlight_{RUN_TAG}_{LOWLIGHT_TAG}"

FULL_CLEAN_BEST_CKPT = (
    CHECKPOINTS_ROOT
    / f"full_clean_detector_{RUN_TAG}_resnet18"
    / f"kaggle_v1_clean_full_{RUN_TAG}_best.pt"
)

LOWLIGHT_BEST_CKPT = (
    CHECKPOINTS_ROOT
    / f"detector_lowlight_{RUN_TAG}_{LOWLIGHT_TAG}_resnet18"
    / "detector_lowlight_best.pt"
)

MIXED_BEST_CKPT = (
    CHECKPOINTS_ROOT
    / f"detector_mixed_{RUN_TAG}_{LOWLIGHT_TAG}_resnet18"
    / "detector_mixed_best.pt"
)

SEED42_COMPARE_ROOT = KAGGLE_V1_ROOT / f"eval_final_model_compare_{RUN_TAG}_{LOWLIGHT_TAG}_5k"
NEW_COMPARE_ROOT = KAGGLE_V1_ROOT / f"eval_final_model_compare_{RUN_TAG}_{LOWLIGHT_TAG}_5k_seed{NEW_SUBSET_SEED}"

CLEAN_MODEL_REPORT_DIR = NEW_COMPARE_ROOT / "clean_detector"
LOWLIGHT_MODEL_REPORT_DIR = NEW_COMPARE_ROOT / "lowlight_detector"
MIXED_MODEL_REPORT_DIR = NEW_COMPARE_ROOT / "mixed_detector"
NEW_FINAL_COMPARISON_CSV = NEW_COMPARE_ROOT / "final_model_comparison_table.csv"

print("TEST_CLEAN_ROOT:", TEST_CLEAN_ROOT)
print("TEST_LOWLIGHT_ROOT:", TEST_LOWLIGHT_ROOT)
print("FULL_CLEAN_BEST_CKPT:", FULL_CLEAN_BEST_CKPT)
print("LOWLIGHT_BEST_CKPT:", LOWLIGHT_BEST_CKPT)
print("MIXED_BEST_CKPT:", MIXED_BEST_CKPT)
print("SEED42_COMPARE_ROOT:", SEED42_COMPARE_ROOT)
print("NEW_COMPARE_ROOT:", NEW_COMPARE_ROOT)
print("NEW_SUBSET_SEED:", NEW_SUBSET_SEED)


TEST_CLEAN_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_clean_subject_class_balanced_20k
TEST_LOWLIGHT_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/test_lowlight_subject_class_balanced_20k_eye_mid
FULL_CLEAN_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/full_clean_detector_subject_class_balanced_20k_resnet18/kaggle_v1_clean_full_subject_class_balanced_20k_best.pt
LOWLIGHT_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_lowlight_subject_class_balanced_20k_eye_mid_resnet18/detector_lowlight_best.pt
MIXED_BEST_CKPT: /content/drive/MyDrive/task_driven_checkpoints/kaggle_v1/detector_mixed_subject_class_balanced_20k_eye_mid_resnet18/detector_mixed_best.pt
SEED42_COMPARE_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k
NEW_COMPARE_ROOT: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compa

In [6]:
!find "{TEST_CLEAN_ROOT}/open" -type f | wc -l
!find "{TEST_CLEAN_ROOT}/closed" -type f | wc -l

!find "{TEST_LOWLIGHT_ROOT}/open" -type f | wc -l
!find "{TEST_LOWLIGHT_ROOT}/closed" -type f | wc -l


8644
8644
8644
8644


In [7]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {FULL_CLEAN_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --lowlight-root {TEST_LOWLIGHT_ROOT} \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --max-total-samples {MAX_TOTAL_SAMPLES} \
  --subset-seed {NEW_SUBSET_SEED} \
  --output-dir {CLEAN_MODEL_REPORT_DIR}


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /content/Task-driven-low-light-enhancement/artifacts/torch_cache/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 174MB/s]
Clean: 100% 79/79 [18:35<00:00, 14.12s/batch]
Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Using balanced subset with 5000 samples total: {'open': 2500, 'closed': 2500}
Clean    loss: 0.1283 | accuracy: 0.9216 | precision: 0.8685 | recall: 0.9936 | f1: 0.9269 | closed_recall: 0.9936 | threshold: 0.70
Saved subset manifest to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/clean_detector/subset_manifest.csv
LowLight: 100% 79/79 [32:34<00:00, 24.74s/batch]
LowLight loss: 0.0429 | accuracy: 0.8886 | precision: 0.9793 | recall: 0.7940 | f1: 0.8770 | closed_recall: 0.7940 | threshold: 0.70
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_

In [8]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {LOWLIGHT_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --lowlight-root {TEST_LOWLIGHT_ROOT} \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --max-total-samples {MAX_TOTAL_SAMPLES} \
  --subset-seed {NEW_SUBSET_SEED} \
  --output-dir {LOWLIGHT_MODEL_REPORT_DIR}


Clean: 100% 79/79 [01:31<00:00,  1.16s/batch]
Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Using balanced subset with 5000 samples total: {'open': 2500, 'closed': 2500}
Clean    loss: 0.3736 | accuracy: 0.5672 | precision: 0.5360 | recall: 1.0000 | f1: 0.6979 | closed_recall: 1.0000 | threshold: 0.40
Saved subset manifest to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/lowlight_detector/subset_manifest.csv
LowLight: 100% 79/79 [00:30<00:00,  2.58batch/s]
LowLight loss: 0.0608 | accuracy: 0.8870 | precision: 0.8190 | recall: 0.9936 | f1: 0.8979 | closed_recall: 0.9936 | threshold: 0.40
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/lowlight_detector

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Clean | 56.72% | 53.60% | 100.00% | 6

In [9]:
!python3 /content/Task-driven-low-light-enhancement/evaluate_transfer_detector.py \
  {MIXED_BEST_CKPT} \
  {TEST_CLEAN_ROOT} \
  --lowlight-root {TEST_LOWLIGHT_ROOT} \
  --batch-size {BATCH_SIZE} \
  --num-workers {NUM_WORKERS} \
  --max-total-samples {MAX_TOTAL_SAMPLES} \
  --subset-seed {NEW_SUBSET_SEED} \
  --output-dir {MIXED_MODEL_REPORT_DIR}


Clean: 100% 79/79 [01:30<00:00,  1.14s/batch]
Evaluating on device: cuda
Classes: {'open': 0, 'closed': 1}
Using balanced subset with 5000 samples total: {'open': 2500, 'closed': 2500}
Clean    loss: 0.0416 | accuracy: 0.9344 | precision: 0.8946 | recall: 0.9848 | f1: 0.9375 | closed_recall: 0.9848 | threshold: 0.50
Saved subset manifest to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/mixed_detector/subset_manifest.csv
LowLight: 100% 79/79 [00:30<00:00,  2.61batch/s]
LowLight loss: 0.0408 | accuracy: 0.9360 | precision: 0.8935 | recall: 0.9900 | f1: 0.9393 | closed_recall: 0.9900 | threshold: 0.50
Saved evaluation report to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/mixed_detector

Report Table
| Dataset | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|
| Clean | 93.44% | 89.46% | 98.48% | 93.75% |

In [10]:
import pandas as pd

clean_manifest = pd.read_csv(CLEAN_MODEL_REPORT_DIR / "subset_manifest.csv")
lowlight_manifest = pd.read_csv(LOWLIGHT_MODEL_REPORT_DIR / "subset_manifest.csv")
mixed_manifest = pd.read_csv(MIXED_MODEL_REPORT_DIR / "subset_manifest.csv")

print("Clean subset rows:", len(clean_manifest))
print("Low-light subset rows:", len(lowlight_manifest))
print("Mixed subset rows:", len(mixed_manifest))
print()
print("Clean == Low-light subset:", clean_manifest.equals(lowlight_manifest))
print("Clean == Mixed subset:", clean_manifest.equals(mixed_manifest))
print("Low-light == Mixed subset:", lowlight_manifest.equals(mixed_manifest))

assert clean_manifest.equals(lowlight_manifest)
assert clean_manifest.equals(mixed_manifest)


Clean subset rows: 5000
Low-light subset rows: 5000
Mixed subset rows: 5000

Clean == Low-light subset: True
Clean == Mixed subset: True
Low-light == Mixed subset: True


In [11]:
clean_report = pd.read_csv(CLEAN_MODEL_REPORT_DIR / "experiment_results.csv")
lowlight_report = pd.read_csv(LOWLIGHT_MODEL_REPORT_DIR / "experiment_results.csv")
mixed_report = pd.read_csv(MIXED_MODEL_REPORT_DIR / "experiment_results.csv")

print("=== Clean Detector ===")
print(clean_report.to_string(index=False))
print()

print("=== Low-Light Detector ===")
print(lowlight_report.to_string(index=False))
print()

print("=== Mixed Detector ===")
print(mixed_report.to_string(index=False))


=== Clean Detector ===
  Dataset  Accuracy  Precision  Recall    F1
    Clean     92.16      86.85   99.36 92.69
Low-light     88.86      97.93   79.40 87.70

=== Low-Light Detector ===
  Dataset  Accuracy  Precision  Recall    F1
    Clean     56.72       53.6  100.00 69.79
Low-light     88.70       81.9   99.36 89.79

=== Mixed Detector ===
  Dataset  Accuracy  Precision  Recall    F1
    Clean     93.44      89.46   98.48 93.75
Low-light     93.60      89.35   99.00 93.93


In [12]:
clean_detailed = pd.read_csv(CLEAN_MODEL_REPORT_DIR / "evaluation_results.csv")
lowlight_detailed = pd.read_csv(LOWLIGHT_MODEL_REPORT_DIR / "evaluation_results.csv")
mixed_detailed = pd.read_csv(MIXED_MODEL_REPORT_DIR / "evaluation_results.csv")

clean_detailed["model"] = "Clean detector"
lowlight_detailed["model"] = "Low-light detector"
mixed_detailed["model"] = "Mixed detector"

detailed = pd.concat(
    [clean_detailed, lowlight_detailed, mixed_detailed],
    ignore_index=True,
)

display(
    detailed[
        ["model", "dataset", "accuracy", "precision", "recall", "f1", "closed_recall", "threshold"]
    ].round(4)
)


,model,dataset,accuracy,precision,recall,f1,closed_recall,threshold
0,Clean detector,Clean,0.9216,0.8685,0.9936,0.9269,0.9936,0.7
1,Clean detector,Low-light,0.8886,0.9793,0.7940,0.8770,0.7940,0.7
2,Low-light detector,Clean,0.5672,0.5360,1.0000,0.6979,1.0000,0.4
3,Low-light detector,Low-light,0.8870,0.8190,0.9936,0.8979,0.9936,0.4
4,Mixed detector,Clean,0.9344,0.8946,0.9848,0.9375,0.9848,0.5
5,Mixed detector,Low-light,0.9360,0.8935,0.9900,0.9393,0.9900,0.5


In [13]:
def row_for(df, model_name, dataset_name):
    row = df[(df["model"] == model_name) & (df["dataset"] == dataset_name)]
    assert len(row) == 1, f"Expected one row for {model_name} / {dataset_name}"
    return row.iloc[0]

summary_rows = []
for model_name in ["Clean detector", "Low-light detector", "Mixed detector"]:
    clean_row = row_for(detailed, model_name, "Clean")
    lowlight_row = row_for(detailed, model_name, "Low-light")
    summary_rows.append({
        "Model": model_name,
        "Clean Accuracy": round(100.0 * clean_row["accuracy"], 2),
        "Clean F1": round(100.0 * clean_row["f1"], 2),
        "Clean Closed Recall": round(100.0 * clean_row["closed_recall"], 2),
        "Low-light Accuracy": round(100.0 * lowlight_row["accuracy"], 2),
        "Low-light F1": round(100.0 * lowlight_row["f1"], 2),
        "Low-light Closed Recall": round(100.0 * lowlight_row["closed_recall"], 2),
        "Avg F1": round(50.0 * (clean_row["f1"] + lowlight_row["f1"]), 2),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

summary_df.to_csv(NEW_FINAL_COMPARISON_CSV, index=False)
print("Saved new final comparison table to:", NEW_FINAL_COMPARISON_CSV)


,Model,Clean Accuracy,Clean F1,Clean Closed Recall,Low-light Accuracy,Low-light F1,Low-light Closed Recall,Avg F1
0,Clean detector,92.16,92.69,99.36,88.86,87.70,79.40,90.19
1,Low-light detector,56.72,69.79,100.00,88.70,89.79,99.36,79.79
2,Mixed detector,93.44,93.75,98.48,93.60,93.93,99.00,93.84


Saved new final comparison table to: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/final_model_comparison_table.csv


In [14]:
seed42_csv = SEED42_COMPARE_ROOT / "final_model_comparison_table.csv"
seed_new_csv = NEW_FINAL_COMPARISON_CSV

print("seed42_csv exists:", seed42_csv.exists())
print("seed_new_csv exists:", seed_new_csv.exists())

assert seed42_csv.exists(), f"Missing old seed-42 summary: {seed42_csv}"
assert seed_new_csv.exists(), f"Missing new seed summary: {seed_new_csv}"

seed42_df = pd.read_csv(seed42_csv)
seed42_df["Subset Seed"] = 42

seed_new_df = pd.read_csv(seed_new_csv)
seed_new_df["Subset Seed"] = NEW_SUBSET_SEED

combined = pd.concat([seed42_df, seed_new_df], ignore_index=True)

print("=== Per-seed Comparison ===")
display(combined)


seed42_csv exists: True
seed_new_csv exists: True
=== Per-seed Comparison ===


,Model,Clean Accuracy,Clean F1,Clean Closed Recall,Low-light Accuracy,Low-light F1,Low-light Closed Recall,Avg F1,Subset Seed
0,Clean detector,92.56,93.04,99.44,89.60,88.56,80.48,90.80,42
1,Low-light detector,57.04,69.95,100.00,88.84,89.87,99.00,79.91,42
2,Mixed detector,93.60,93.90,98.60,93.72,94.02,98.72,93.96,42
3,Clean detector,92.16,92.69,99.36,88.86,87.70,79.40,90.19,314
4,Low-light detector,56.72,69.79,100.00,88.70,89.79,99.36,79.79,314
5,Mixed detector,93.44,93.75,98.48,93.60,93.93,99.00,93.84,314


In [15]:
metric_cols = [
    "Clean Accuracy",
    "Clean F1",
    "Clean Closed Recall",
    "Low-light Accuracy",
    "Low-light F1",
    "Low-light Closed Recall",
    "Avg F1",
]

agg_df = combined.groupby("Model")[metric_cols].agg(["mean", "std"]).round(2)

print("=== Mean / Std Across Two Subsets ===")
display(agg_df)

for seed_value, df_part in combined.groupby("Subset Seed"):
    best_row = df_part.sort_values("Avg F1", ascending=False).iloc[0]
    print(
        f"Best model on seed {seed_value}: "
        f"{best_row['Model']} | Avg F1={best_row['Avg F1']:.2f}"
    )


=== Mean / Std Across Two Subsets ===


Clean Accuracy       Clean F1       Clean Closed Recall  \
                             mean   std     mean   std                mean   
Model                                                                        
Clean detector              92.36  0.28    92.86  0.25               99.40   
Low-light detector          56.88  0.23    69.87  0.11              100.00   
Mixed detector              93.52  0.11    93.82  0.11               98.54   

                         Low-light Accuracy       Low-light F1        \
                     std               mean   std         mean   std   
Model                                                                  
Clean detector      0.06              89.23  0.52        88.13  0.61   
Low-light detector  0.00              88.77  0.10        89.83  0.06   
Mixed detector      0.08              93.66  0.08        93.98  0.06   

                   Low-light Closed Recall       Avg F1        
                                      mean   std   mean   std  
Model                                                          
Clean detector                       79.94  0.76  90.50  0.43  
Low-light detector                   99.18  0.25  79.85  0.08  
Mixed detector                       98.86  0.20  93.90  0.08

Best model on seed 42: Mixed detector | Avg F1=93.96
Best model on seed 314: Mixed detector | Avg F1=93.84


In [16]:
print("=== Clean detector summary ===")
print((CLEAN_MODEL_REPORT_DIR / "evaluation_summary.txt").read_text())
print()

print("=== Low-light detector summary ===")
print((LOWLIGHT_MODEL_REPORT_DIR / "evaluation_summary.txt").read_text())
print()

print("=== Mixed detector summary ===")
print((MIXED_MODEL_REPORT_DIR / "evaluation_summary.txt").read_text())


=== Clean detector summary ===
Using threshold=0.70 for the closed-eye class.
Closed-eye recall on clean data: 0.9936.
Balanced subset used for evaluation: 5000 samples total.
Per-class subset counts: {'open': 2500, 'closed': 2500}.
Closed-eye recall on low-light data: 0.7940.
Low-light degradation summary: accuracy 92.16% -> 88.86%.
Accuracy: -3.30 points (drop).
Precision: +11.07 points (change).
Recall: -19.96 points (drop).
F1: -4.99 points (drop).

=== Low-light detector summary ===
Using threshold=0.40 for the closed-eye class.
Closed-eye recall on clean data: 1.0000.
Balanced subset used for evaluation: 5000 samples total.
Per-class subset counts: {'open': 2500, 'closed': 2500}.
Closed-eye recall on low-light data: 0.9936.
Low-light degradation summary: accuracy 56.72% -> 88.70%.
Accuracy: +31.98 points (change).
Precision: +28.30 points (change).
Recall: -0.64 points (drop).
F1: +20.00 points (change).

=== Mixed detector summary ===
Using threshold=0.50 for the closed-eye clas

In [17]:
print("Fresh comparison notebook complete.")
print("CLEAN_MODEL_REPORT_DIR:", CLEAN_MODEL_REPORT_DIR)
print("LOWLIGHT_MODEL_REPORT_DIR:", LOWLIGHT_MODEL_REPORT_DIR)
print("MIXED_MODEL_REPORT_DIR:", MIXED_MODEL_REPORT_DIR)
print("NEW_FINAL_COMPARISON_CSV:", NEW_FINAL_COMPARISON_CSV)


Fresh comparison notebook complete.
CLEAN_MODEL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/clean_detector
LOWLIGHT_MODEL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/lowlight_detector
MIXED_MODEL_REPORT_DIR: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/mixed_detector
NEW_FINAL_COMPARISON_CSV: /content/drive/MyDrive/task_driven_video_pipeline/kaggle_v1/eval_final_model_compare_subject_class_balanced_20k_eye_mid_5k_seed314/final_model_comparison_table.csv
